# Ticket Preprocessing

This notebook compiles all preprocessing steps taken to prepare data for pipeline experimentation. The task was broken down into 4 steps:
1. Text Cleaning
2. Tokenization and Lemmatization
3. Specific Issue Handling and Ticket Length Filtering

## Step 1: Text Cleaning

In [1]:
#Install
%pip install contractions

Note: you may need to restart the kernel to use updated packages.


In [2]:
#importing libraries
import pandas as pd
import re
import contractions


In [3]:
df = pd.read_csv('/Users/annie/Downloads/customer_support_tickets_en.csv')
df['body'] = df['body'].fillna('')
print(f"Loaded {len(df)} rows")

Loaded 28261 rows


In [4]:
#cleaning funtion
stats = {
    'literal_newlines_fixed': 0,
    'boilerplate_removed':    0,
    'contractions_expanded':  0,
    'hyphens_fixed':          0,
    'lowercased':             0,
    'html_tags_removed': 0,
    'special_chars_removed':  0,
    'whitespace_fixed':       0,
    'empty_after_clean':      0,
}

def clean_text(text):
    global stats
    text = str(text).strip()

    # 1. Fix literal \n
    if '\\n' in text:
        text = text.replace('\\n', ' ')
        stats['literal_newlines_fixed'] += 1

    # 2. Remove boilerplate greetings/sign-offs
    boilerplate_patterns = [
        r'^dear customer support team[,\s]*',
        r'^dear support team[,\s]*',
        r'^dear customer service[,\s]*',
        r'^hello[,\s]*',
        r'^hi[,\s]*',
        r'best regards.*$',
        r'kind regards.*$',
        r'sincerely.*$',
        r'thank you for your (help|assistance|support).*$',
    ]
    original = text
    for pattern in boilerplate_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE | re.MULTILINE)
    if text != original:
        stats['boilerplate_removed'] += 1

    # 3. Expand contractions
    expanded = contractions.fix(text)
    if expanded != text:
        stats['contractions_expanded'] += 1
    text = expanded

    # 4. Fix hyphens — cloud-native → cloud native
    dehyphenated = re.sub(r'([a-zA-Z])-([a-zA-Z])', r'\1 \2', text)
    if dehyphenated != text:
        stats['hyphens_fixed'] += 1
    text = dehyphenated

    # 5. Lowercase
    text = text.lower()
    stats['lowercased'] += 1

    # 6. Remove <br> line breaks
    html_tags_removed = re.sub(r'<.*?>', ' ', text)
    if html_tags_removed != text:
        stats['html_tags_removed'] += 1
    text = html_tags_removed

    # 7. Remove special characters
    cleaned = re.sub(r'[^a-z\s]', '', text)
    if cleaned != text:
        stats['special_chars_removed'] += 1
    text = cleaned

    # 8. Remove extra whitespace
    stripped = re.sub(r'\s+', ' ', text).strip()
    if stripped != text:
        stats['whitespace_fixed'] += 1
    text = stripped

    if text == '':
        stats['empty_after_clean'] += 1

    return text

In [5]:
# apply and print stats
df['clean_body'] = df['body'].apply(clean_text)

print("=" * 55)
print("        CLEANING STATISTICS")
print("=" * 55)
for step, count in stats.items():
    pct = (count / len(df)) * 100
    print(f"  {step:<30} {count:>6}  ({pct:.1f}%)")
print("=" * 55)

# Find rows that had URLs, emails or contractions to verify cleaning worked
test_cases = df[df['body'].str.contains("http|@|<br>|n't|'ve|'ll", na=False)].head(5)

for i, row in test_cases.iterrows():
    print("ORIGINAL:", row['body'][:200])
    print("CLEANED: ", row['clean_body'][:200])
    print("-" * 60)

        CLEANING STATISTICS
  literal_newlines_fixed            997  (3.5%)
  boilerplate_removed              4564  (16.1%)
  contractions_expanded            2885  (10.2%)
  hyphens_fixed                    3522  (12.5%)
  lowercased                      28261  (100.0%)
  html_tags_removed                 983  (3.5%)
  special_chars_removed           27513  (97.4%)
  whitespace_fixed                 5424  (19.2%)
  empty_after_clean                   1  (0.0%)
ORIGINAL: Dear Customer Support Team,\n\nI am reaching out to report persistent issues with network connectivity that are significantly disrupting my workflow. I've observed sporadic interruptions across severa
CLEANED:  i am reaching out to report persistent issues with network connectivity that are significantly disrupting my workflow i have observed sporadic interruptions across several devices which i believe may 
------------------------------------------------------------
ORIGINAL: Customer Service Team,\n\nWe are facing 

In [6]:
# verfying each fix with examples
# 1. Newline fix
print("=== 1. NEWLINE FIX ===")
sample = df[df['body'].str.contains(r'\\n', na=False)].iloc[0]
print("BEFORE:", sample['body'][:150])
print("AFTER: ", sample['clean_body'][:150])

print()
# 2. Hyphen fix
print("=== 2. HYPHEN FIX ===")
sample = df[df['body'].str.contains('cloud-native|real-time', na=False)].iloc[0]
print("BEFORE:", sample['body'][:150])
print("AFTER: ", sample['clean_body'][:150])

print()
# 3. Contraction fix
print("=== 3. CONTRACTION FIX ===")
sample = df[df['body'].str.contains("n't|'ve", na=False)].iloc[0]
print("BEFORE:", sample['body'][:150])
print("AFTER: ", sample['clean_body'][:150])

print()
# 4. Boilerplate removal
print("=== 4. BOILERPLATE REMOVAL ===")
sample = df[df['body'].str.startswith('Dear Customer Support Team', na=False)].iloc[0]
print("BEFORE:", sample['body'][:150])
print("AFTER: ", sample['clean_body'][:150])

print()
#5. HTML Tag Removal
print("=== 5. HTML TAGS REMOVAL ===")
sample = df[df['body'].str.contains('<br>|', na=False)].iloc[0]
print("BEFORE:", sample['body'][:150])
print("AFTER: ", sample['clean_body'][:150])

=== 1. NEWLINE FIX ===
BEFORE: Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to
AFTER:  i am writing to report a significant problem with the centralized account management portal which currently appears to be offline this outage is block

=== 2. HYPHEN FIX ===
BEFORE: Dear Customer Support Team,\n\nI am reaching out to request comprehensive details on optimizing marketing workflows across multiple departments by uti
AFTER:  i am reaching out to request comprehensive details on optimizing marketing workflows across multiple departments by utilizing advanced analytics autom

=== 3. CONTRACTION FIX ===
BEFORE: Dear Customer Support Team,\n\nI am reaching out to report persistent issues with network connectivity that are significantly disrupting my workflow. 
AFTER:  i am reaching out to report persistent issues with network connectivity that are significantly disrupting my workflow i hav

In [7]:
# Word count before vs after
df['word_count_before'] = df['body'].apply(lambda x: len(str(x).split()))
df['word_count_after_cleaning']  = df['clean_body'].apply(lambda x: len(x.split()))

print(f"Avg words BEFORE: {df['word_count_before'].mean():.1f}")
print(f"Avg words AFTER:  {df['word_count_after_cleaning'].mean():.1f}")
print(f"Avg words removed per ticket: {(df['word_count_before'] - df['word_count_after_cleaning']).mean():.1f}")

# Empty or too short after cleaning
print(f"\nEmpty after cleaning:           {(df['clean_body'].str.strip() == '').sum()}")
print(f"Fewer than 3 words after clean: {(df['clean_body'].apply(lambda x: len(x.split()) < 3)).sum()}")

Avg words BEFORE: 55.4
Avg words AFTER:  54.8
Avg words removed per ticket: 0.6

Empty after cleaning:           1
Fewer than 3 words after clean: 54


In [8]:
# saving cleaned data
df.to_csv('/Users/annie/Masters/AI_and_TA/Coursework/Task4/customer_support_tickets_cleaned_final.csv', index=False)
print(f"Saved {len(df)} rows → customer_support_tickets_cleaned_final.csv")

Saved 28261 rows → customer_support_tickets_cleaned_final.csv


## Text Cleaning Summary
**Input:** `customer_support_tickets_en.csv` (28,261 English tickets)  
**Output:** `customer_support_tickets_cleaned_final.csv`

### What this section does:
This section performs text cleaning on the raw `body` column of customer
support tickets. The cleaned text is saved in a new column `clean_body` 
and will be used in all downstream NLP tasks (stopword removal, 
lemmatisation, vectorisation, clustering, topic modelling).

### Cleaning Steps (in order):

| Step | What it does | Why |
|------|-------------|-----|
| 1. Fix literal `\n` | Replaces stored `\n` strings with a space | Prevents words merging e.g. `teamnni` |
| 2. Boilerplate removal | Removes greetings like "Dear Customer Support Team" and sign-offs | These are structural noise, not issue content |
| 3. Contraction expansion | `can't` → `cannot`, `we've` → `we have` | Ensures consistent vocabulary |
| 4. Hyphen fix | `cloud-native` → `cloud native` | Prevents words merging when special chars are removed |
| 5. Lowercase | All text to lowercase | Treats `Network` and `network` as the same word |
| 6. HTML tag removal | Removes all HTML line breaks e.g. <br>| Prevents words merging when special chars are removed |
| 6. Special character removal | Removes punctuation, numbers, symbols | Keeps only alphabetic content for NLP |
| 7. Whitespace normalisation | Collapses multiple spaces into one | Ensures clean token boundaries |

### Results:

| Metric | Value |
|--------|-------|
| Total tickets | 28,261 |
| Literal `\n` fixed | 997 (3.5%) |
| Boilerplate removed | ~4,800+ tickets |
| Contractions expanded | 696 (2.5%) |
| Hyphens fixed | 3,571 (12.6%) |
| HTML tags removed | 983 (3.5%)
| Special chars removed | 27,517 (97.4%) |
| Avg words before cleaning | 55.4 |
| Avg words after cleaning | 54.8 |
| Empty after cleaning | 1 |
| Fewer than 3 words | 54 |

### Notes:
- No emails or URLs were found in this dataset so those steps had no effect
- The 1 empty ticket and 54 very short tickets are retained for downstream preprocessing

## Step 2: Tokenization and Lemmatization

This section performs tokenization, stopword removal, and lemmatization on cleaned text.

In [9]:
import pandas as pd
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.stem import WordNetLemmatizer
import nltk
import pandas as pd
from langdetect import detect

As preprocessing was continued, we found that some German words were still in the dataset, despite initial filtering using `df=df[df['language'] == 'en']`. This may be a result of translation issues or classification errors in the original multilingual dataset (which may have affected initial language filtering). As a result we completed secondary filtering on remaining rows.

In [10]:
df = pd.read_csv('/Users/annie/Masters/AI_and_TA/Coursework/Task4/customer_support_tickets_cleaned_final.csv')
df['clean_body'] = df['clean_body'].fillna('')
print(f'Loaded {len(df)} rows')
print(f'Before language filter: {len(df)} rows')
def is_english(text):
    try:
        return detect(str(text)) == "en"
    except:
        return False

df["is_en_text"] = df["clean_body"].apply(is_english)
df = df[df["is_en_text"]].copy()
df.drop(columns=["is_en_text"], inplace=True)
print(f'After language filter: {len(df)} rows')

Loaded 28261 rows
Before language filter: 28261 rows
After language filter: 28122 rows


## Tokenization

In [11]:
stop_words = set(ENGLISH_STOP_WORDS)

custom_stopwords = {
    'dear', 'customer', 'support', 'team',
    'hello', 'hi', 'thanks', 'thank',
    'regards', 'please', 'kindly',
    'hope', 'message'}

def tokenize_text(text):
    if text is None:
        return []
    tokens = text.split()
    tokens = [word for word in tokens
              if word not in stop_words
              and word not in custom_stopwords
              and len(word) > 2]
    tokens = list(dict.fromkeys(tokens))
    return tokens

df['tokens'] = df['clean_body'].apply(tokenize_text)
df[['clean_body', 'tokens']].head()

,clean_body,tokens
0,i am writing to report a significant problem w...,"[writing, report, significant, problem, centra..."
1,i hope this message reaches you well i am reac...,"[reaches, reaching, request, detailed, informa..."
2,i hope this message finds you well i am reachi...,"[finds, reaching, request, clarification, bill..."
3,i hope this message reaches you well i am reac...,"[reaches, reaching, ask, compatibility, produc..."
4,dear customer support i hope this message reac...,"[reaches, good, health, eager, learn, features..."


## Lemmatization

In [12]:
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    result = []
    for word in tokens:
        word = lemmatizer.lemmatize(word, 'v')
        word = lemmatizer.lemmatize(word, 'n')
        result.append(word)
    result = list(dict.fromkeys(result))
    return result

df['processed_tokens'] = df['tokens'].apply(lemmatize_tokens)
df['processed_text'] = df['processed_tokens'].apply(lambda x: ' '.join(x))

df = df[df['processed_text'].str.strip() != ''].reset_index(drop=True)

df[['tokens', 'processed_tokens', 'processed_text']].head()

,tokens,processed_tokens,processed_text
0,"[writing, report, significant, problem, centra...","[write, report, significant, problem, centrali...",write report significant problem centralize ac...
1,"[reaches, reaching, request, detailed, informa...","[reach, request, detail, information, capabili...",reach request detail information capability sm...
2,"[finds, reaching, request, clarification, bill...","[find, reach, request, clarification, bill, pa...",find reach request clarification bill payment ...
3,"[reaches, reaching, ask, compatibility, produc...","[reach, ask, compatibility, product, specific,...",reach ask compatibility product specific need ...
4,"[reaches, good, health, eager, learn, features...","[reach, good, health, eager, learn, feature, p...",reach good health eager learn feature product ...


In [15]:
df['word_count_after_t_and_l'] = df['processed_text'].apply(lambda x: len(x.split()))
df[['word_count_after_cleaning','word_count_after_t_and_l']].head()

print(f'Average ticket length after processing:{df['word_count_after_t_and_l'].sum()/len(df):2f}')
print(f'Words removed during processing: {df['word_count_after_cleaning'].sum()-df['word_count_after_t_and_l'].sum()}')

Average ticket length after processing:26.589225
Words removed during processing: 799241
747689


In [14]:
# Save tokenized and lemmatized data
df.to_csv('/Users/annie/Masters/AI_and_TA/Coursework/Task4/tokenized_and_lemmatized_tickets_final.csv', index=False)
print(f'Saved {len(df)} rows to tokenized_and_lemmatized_tickets_final.csv')

Saved 28120 rows to tokenized_and_lemmatized_tickets_final.csv


### Tokenization and Lemmatization Summary
**Input:** `customer_support_tickets_cleaned_final.csv`

**Output:** `tokenized_and_lemmatized_tickets_final.csv`

### Results:

| Metric                                 | Value           |
|----------------------------------------|-----------------|
| Total tickets before processing        | 28,261          |
| Total tickets after processing         | 28,120          |
| Avg words per ticket before processing | 54.8            |
| Avg words per ticket after processing  | 26.59           |
| Max ticket length after processing     | 107.0           |
| Min ticket length after processing     | 1.0             |
| Words removed during processing        | 799237 (51.67%) |

## Step 3: Issue Handling and Ticket Length Filtering

This notebook takes tickets which have already been cleaned, tokenized and lemmatized, and handles specific issues such as ticket IDs, customer names, dates and product codes, as well as filtering out very short tickets and truncating very long tickets.

In [ ]:
import pandas as pd
import enchant
from collections import Counter
from langdetect import detect
import matplotlib.pyplot as plt

### Load cleaned and lemmatized tokens

In [ ]:
df = pd.read_csv('/Users/annie/Masters/AI_and_TA/Coursework/Task4/tokenized_and_lemmatized_tickets_final.csv')
print(f'Loaded {len(df)} tickets.')
df.head()

In [ ]:
# Load processed text column to assess obvious issues which need handling.
pd.set_option('display.max_colwidth', None)
df['processed_text']

### Handling Special Cases

In previous preprocessing steps, special characters ( including hyphens and numerical characters) have been removed. All text has also been converted to lower-case. This means that names, ticket IDs and product codes  have already been reduced to meaningless character sequences, making them harder to identify. Purely alphabetic sequences could survive the cleaning process, however manual inspection of tokens in the `processed_text` which aren't in the English Dictionary can help us reduce the effect on downstream analysis.

In [ ]:
all_tokens = []
for text in df['processed_text'].dropna():
    all_tokens.extend(text.split())

en = enchant.Dict("en_GB")
us = enchant.Dict("en_US")
non_eng = []
non_eng_count = 0
counts = Counter(all_tokens)

for word, count in counts.most_common():
    if not en.check(word):
        if not us.check(word):
           non_eng.append((word, count))
           non_eng_count += count

print(f"Tokens not found in English dictionary:{non_eng_count}")
for word, count in non_eng:
    print(f" {word: <15} {count}")

print(f'Proportion of tokens not in English Dictionary:{non_eng_count/len(all_tokens)*100:.2f}%')

In [ ]:
uncommon_non_eng_tokens = 0
for words, count in non_eng:
    if count < 100:
        uncommon_non_eng_tokens += count
print(f'Total uncommon non-English tokens: {uncommon_non_eng_tokens}')

print(f'As proportion of all tokens:{uncommon_non_eng_tokens / sum(counts.values()):.2f}%')

Within the `non-eng` words, the most frequently appearing are company names and domain specific vocabulary, such as abbreviations. There is no evidence of consistent patterns of customer names, formatted ticket IDs, or product codes being left as tokens in the dataset. Many of the flagged words appear <100 times within the dataset, and calculating the sum of these as a proportion of all tokens shows that they will have minimal impact on clustering or topic modelling downstream. Those appearing more than 100 times can all be manually observed as company names and domain specific vocabulary, so should be retained, as they may be beneficial for gaining insights later in the project. As a result we will consider no special cases to have been found and will retain all `non-eng` tokens to avoid over-cleaning.

### Filtering by Ticket Length

In [ ]:
# Identify tickets which are too short after cleaning
short_tickets = df['processed_text'].apply(lambda x: len(x.split()) <= 3)
print(f'Number of processed tickets with 3 words or fewer: {short_tickets.sum()} ')
print(f'Tokens removed: {df.loc[short_tickets,'word_count_after_t_and_l'].sum()}')
pd.set_option('display.max_colwidth', 50)
df[short_tickets]

In [ ]:
# Remove short tickets from dataset
df_filtered= df[~short_tickets].copy()
df = df_filtered.reset_index(drop=True)

print(f'Tickets of length <=3 have been removed: {len(df)} tickets remaining ')

In [ ]:
# Identify upper tickets length outliers
df['word_count_after_t_and_l'].describe()

In [ ]:
# Identify cutoff length
max_words = df['word_count_after_t_and_l'].quantile(0.99)
print(f'99th percentile word count of after preprocessing: {max_words}')

In [ ]:
# Calculate proportion of tickets that exceed cutoff
above_cutoff = df[df['word_count_after_t_and_l'] > max_words]
print(f'Number of tickets above 99th percentile:{len(above_cutoff)}')
print(f'Proportion of tickets above 99th percentile:{len(above_cutoff)/len(df):.4f}%')

# Plot ticket distribution above cutoff
above_cutoff['word_count_after_t_and_l'].hist(bins=20)
plt.xlabel('Word Count')
plt.ylabel('Number of Tickets')
plt.title('Distribution of Long Tickets (Above 99th Percentile)')

plt.grid(False)
plt.tight_layout()
plt.show()

We can see that within those tickets above the 99th percentile cutoff, there is a clear sparsity of tickets over 85 words in length. It therefore makes sense to up the cutoff to 85 words (to not lose valuable data). Those tickets above 85 words can be truncated to not lose all information.

In [ ]:
# Apply truncation
df['trunc_tokens'] = df['processed_text'].apply(lambda x: x.split()[:85])
df['trunc_processed_text'] = df['trunc_tokens'].apply(lambda x: ' '.join(x))

df.head()

In [ ]:
# Calculate Average Ticket Length
df['word_count_final'] = df['trunc_processed_text'].apply(lambda x: len(x.split()))
avg_word_count= (df['word_count_final'].sum())/len(df)
print(f'After preprocessing average ticket length is {avg_word_count:.2f} words.')

In [ ]:
# Save tokenized and lemmatized data
df.to_csv('/Users/annie/Masters/AI_and_TA/Coursework/Task4/complete_preprocessing_final.csv', index=False)
print(f'Saved {len(df)} rows to complete_preprocessing_final.csv')

### Issue Handling and Length Filtering Summary
**Input:** `tokenized_and_lemmatized_tickets_final.csv`

**Output:** `complete_preprocessing_final.csv`

### Results:

| Metric                                                         | Value          |
|----------------------------------------------------------------|----------------|
| Total tickets before filtering                                 | 28,118         |
| Total tickets after filtering                                  | 27,921         |
| Avg ticket length before filtering                             | 26.59          |
| Avg ticket length after filtering                              | 26.76          |
| Max ticket length after filtering                              | 85.0           |
| Min ticket length after filtering                              | 4.0            |
| Tokens removed during filtering (Short tickets and Truncation) | 557.0 (0.074%) |